# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [5]:
import pandas as pd

# Seleccionar el DataFrame de DIVIPOLA (usamos 'divipola' o 'df' según tu entorno)
df_base = divipola if 'divipola' in locals() else df

# 1. Crear 'codigo_municipio' asegurando 5 dígitos numéricos (completando ceros a la izquierda si aplica)
df_base['codigo_municipio'] = df_base['Código Municipio'].astype(str).str.strip().str.zfill(5)

# 2. Comprobar que todos los códigos tengan exactamente 5 dígitos numéricos y sean únicos
cumple_formato = df_base['codigo_municipio'].str.fullmatch(r"[0-9]{5}").all()
es_unico = df_base['codigo_municipio'].is_unique
sin_vacios = (df_base['codigo_municipio'].isna().sum() == 0) and not df_base['codigo_municipio'].eq('').any()

print("--- VALIDACIONES DE CÓDIGO MUNICIPIO ---")
print(f"¿Todos los códigos tienen 5 dígitos numéricos?: {cumple_formato}")
print(f"¿El código de municipio es único?:               {es_unico}")
print(f"¿Está libre de nulos o vacíos?:                  {sin_vacios}")

# 3. Contar los valores por clasificación territorial usando la columna exacta
col_tipo = 'Tipo: Municipio / Isla / Área no municipalizada'
conteo_tipos = df_base[col_tipo].value_counts()

print("\n--- DISTRIBUCIÓN POR TIPO TERRITORIAL ---")
print(conteo_tipos)

# 4. Construir la dimensión territorial con las columnas solicitadas
dimension = df_base[[
    'codigo_municipio',
    'Nombre Departamento',
    'Nombre Municipio',
    col_tipo
]].copy()

# Renombrar columnas para estandarización
dimension.columns = [
    'codigo_municipio',
    'nombre_departamento',
    'nombre_municipio',
    'clasificacion_territorial'
]

print("\n--- VISTA PREVIA DE LA DIMENSIÓN TERRITORIAL ---")
display(dimension.head(10))

# 5. Comprobaciones obligatorias (Asserts)
assert len(dimension) == 1122, f"Se esperaban 1.122 entidades, pero se encontraron {len(dimension):,}"
assert dimension['codigo_municipio'].is_unique, "Error: Existen códigos de municipio duplicados."
assert dimension['codigo_municipio'].str.fullmatch(r"[0-9]{5}").all(), "Error: Existen códigos sin 5 dígitos."

print("\n✅ COMPROBACIÓN EXITOSA: La dimensión conserva exactamente las 1.122 entidades, los 3 tipos territoriales y un código único de 5 dígitos.")

--- VALIDACIONES DE CÓDIGO MUNICIPIO ---
¿Todos los códigos tienen 5 dígitos numéricos?: True
¿El código de municipio es único?:               True
¿Está libre de nulos o vacíos?:                  True

--- DISTRIBUCIÓN POR TIPO TERRITORIAL ---
Tipo: Municipio / Isla / Área no municipalizada
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64

--- VISTA PREVIA DE LA DIMENSIÓN TERRITORIAL ---


,codigo_municipio,nombre_departamento,nombre_municipio,clasificacion_territorial
0,05001,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05021,ANTIOQUIA,ALEJANDRÍA,Municipio
4,05030,ANTIOQUIA,AMAGÁ,Municipio
5,05031,ANTIOQUIA,AMALFI,Municipio
6,05034,ANTIOQUIA,ANDES,Municipio
7,05036,ANTIOQUIA,ANGELÓPOLIS,Municipio
8,05038,ANTIOQUIA,ANGOSTURA,Municipio
9,05040,ANTIOQUIA,ANORÍ,Municipio



✅ COMPROBACIÓN EXITOSA: La dimensión conserva exactamente las 1.122 entidades, los 3 tipos territoriales y un código único de 5 dígitos.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [6]:
import pandas as pd

# 1. Preparar 'codigo_municipio' y 'anio' en el DataFrame 'eva'
# Se limpia espacios sin perder los valores originales
cod_muni_limpio = eva['Código Dane municipio'].astype(str).str.strip()

# Identificar códigos numéricos válidos (entre 1 y 5 dígitos numéricos)
es_valido = cod_muni_limpio.str.fullmatch(r"[0-9]{1,5}")

# Completar ceros a la izquierda solo en los textos numéricos válidos; mantener original/nulo si no es válido
eva['codigo_municipio'] = cod_muni_limpio.where(~es_valido, cod_muni_limpio.str.zfill(5))

# Preparación de 'anio' en eva conservando la información
eva['anio'] = pd.to_numeric(eva['Año'].astype(str).str.strip().str.replace(',', '', regex=False), errors='coerce').astype('Int64')

# 2. Extraer combinaciones distintas de (codigo_municipio, anio) desde EVA
llaves_eva = eva[['codigo_municipio', 'anio']].drop_duplicates().reset_index(drop=True)

print(f"Total combinaciones distintas (codigo_municipio, anio) en EVA: {len(llaves_eva):,}")

# 3. Realizar Anti-Join frente a la dimensión territorial por 'codigo_municipio'
# La unión es many_to_one (múltiples registros/años en EVA por cada municipio en la dimensión)
cruce = llaves_eva.merge(
    dimension[['codigo_municipio']],
    on='codigo_municipio',
    how='left',
    indicator=True,
    validate='many_to_one'
)

# Filtrar únicamente las llaves que no tuvieron correspondencia en 'dimension'
sin_correspondencia = cruce.query("_merge == 'left_only'").drop(columns=['_merge']).reset_index(drop=True)

# 4. Mostrar el diagnóstico resultante (incluso si está vacío)
print("\n--- REGISTROS DE EVA SIN CORRESPONDENCIA EN DIVIPOLA ---")
print(f"Total combinaciones sin correspondencia: {len(sin_correspondencia):,}")
display(sin_correspondencia)

# 5. Comprobaciones obligatorias
print("\n✅ COMPROBACIÓN EXITOSA:")
print(f"- Se preservó el año en el diagnóstico de llaves.")
print(f"- Se utilizó exclusivamente 'codigo_municipio' como clave de unión (validate='many_to_one').")

Total combinaciones distintas (codigo_municipio, anio) en EVA: 7,691

--- REGISTROS DE EVA SIN CORRESPONDENCIA EN DIVIPOLA ---
Total combinaciones sin correspondencia: 0


,codigo_municipio,anio



✅ COMPROBACIÓN EXITOSA:
- Se preservó el año en el diagnóstico de llaves.
- Se utilizó exclusivamente 'codigo_municipio' como clave de unión (validate='many_to_one').


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [7]:
import pandas as pd

# 1. Asegurar la conversión de 'Área cosechada' y 'Producción' a tipo flotante
eva['area_ha'] = pd.to_numeric(
    eva['Área cosechada'].astype(str).str.strip().str.replace(',', '.', regex=False), 
    errors='coerce'
)
eva['produccion_t'] = pd.to_numeric(
    eva['Producción'].astype(str).str.strip().str.replace(',', '.', regex=False), 
    errors='coerce'
)

# 2. Filtrar únicamente los registros de Cultivo == "Papa" y año 2024
cond_papa_2024 = eva['Cultivo'].str.strip().eq("Papa") & eva['anio'].eq(2024)
papa_2024 = eva.loc[cond_papa_2024].copy()

# 3. Identificar filas aptas para el indicador (área cosechada > 0 y producción >= 0)
cond_apta = papa_2024['area_ha'].gt(0) & papa_2024['produccion_t'].ge(0)

# 4. Separar conjuntos: 'apta' y filas excluidas
apta = papa_2024.loc[cond_apta].copy()
excluidas = papa_2024.loc[~cond_apta].copy()

# 5. Conteos y comprobación de conciliación
total_papa_2024 = len(papa_2024)
total_apta = len(apta)
total_excluidas = len(excluidas)

print("--- RESULTADOS DE SELECCIÓN PARA EL INDICADOR (PAPA 2024) ---")
print(f"Total registros Papa 2024:    {total_papa_2024:,}")
print(f"Filas aptas para indicador:   {total_apta:,}")
print(f"Filas excluidas del indicador:{total_excluidas:,}")

# 6. Comprobaciones obligatorias (Asserts)
assert total_papa_2024 == total_apta + total_excluidas, "Error: La suma de aptas y excluidas no coincide con el total de papa_2024."
assert (apta['area_ha'] > 0).all(), "Error: Existen filas aptas con área cosechada <= 0 o nula."
assert (apta['produccion_t'] >= 0).all(), "Error: Existen filas aptas con producción < 0 o nula."

print("\n✅ COMPROBACIÓN EXITOSA: El total de papa_2024 concilia exactamente con las filas aptas más las excluidas.")

--- RESULTADOS DE SELECCIÓN PARA EL INDICADOR (PAPA 2024) ---
Total registros Papa 2024:    831
Filas aptas para indicador:   779
Filas excluidas del indicador:52

✅ COMPROBACIÓN EXITOSA: El total de papa_2024 concilia exactamente con las filas aptas más las excluidas.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [8]:
import pandas as pd

# 1. Agrupar el DataFrame 'apta' por las claves requeridas
# Se especifica as_index=False y dropna=False para conservar la estructura tabular y los grupos con nulos si los hubiera
rendimiento = (
    apta.groupby(
        ['codigo_municipio', 'Cultivo', 'Estado físico del cultivo', 'anio'],
        as_index=False,
        dropna=False
    )
    .agg(
        registros=('area_ha', 'size'),
        produccion_t=('produccion_t', 'sum'),
        area_ha=('area_ha', 'sum')
    )
)

# 2. Calcular el rendimiento agregado (toneladas por hectárea)
# rendimiento_t_ha = suma_produccion / suma_area
rendimiento['rendimiento_t_ha'] = rendimiento['produccion_t'] / rendimiento['area_ha']

# 3. Mostrar vista previa del resultado
print("--- RESUMEN DE RENDIMIENTO AGREGADO (PAPA 2024) ---")
display(rendimiento.head(10))

# 4. Comprobaciones obligatorias (Asserts)
total_registros_rendimiento = rendimiento['registros'].sum()
total_apta = len(apta)

print(f"\nSuma de registros en 'rendimiento': {total_registros_rendimiento:,}")
print(f"Total filas en 'apta':              {total_apta:,}")

# Conciliación de registros totales
assert total_registros_rendimiento == total_apta, "Error: La suma de los conteos por grupo no coincide con len(apta)."

# Comprobar que cada grupo tiene un área total estrictamente positiva
assert (rendimiento['area_ha'] > 0).all(), "Error: Existen grupos agregados con área total no positiva."

# Comprobar que el cálculo de rendimiento no generó infinitos ni nulos
assert not rendimiento['rendimiento_t_ha'].isna().any(), "Error: Se detectaron valores NaN en el rendimiento."
assert not (rendimiento['rendimiento_t_ha'] == float('inf')).any(), "Error: Se detectaron divisiones por cero o valores infinitos."

print("\n✅ COMPROBACIÓN EXITOSA: La suma de conteos coincide con len(apta), cada grupo tiene área total positiva y el indicador 'rendimiento_t_ha' fue calculado correctamente.")

--- RESUMEN DE RENDIMIENTO AGREGADO (PAPA 2024) ---


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t,area_ha,rendimiento_t_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459
5,05088,Papa,En fresco,2024,2,1710.0,95.0,18.000000
6,05093,Papa,En fresco,2024,1,6.0,2.0,3.000000
7,05148,Papa,En fresco,2024,4,14757.0,789.0,18.703422
8,05206,Papa,En fresco,2024,2,1200.0,80.0,15.000000
9,05237,Papa,En fresco,2024,3,2638.0,103.8,25.414258



Suma de registros en 'rendimiento': 779
Total filas en 'apta':              779

✅ COMPROBACIÓN EXITOSA: La suma de conteos coincide con len(apta), cada grupo tiene área total positiva y el indicador 'rendimiento_t_ha' fue calculado correctamente.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [9]:
import pandas as pd

# 1. Guardar la cantidad de filas previas para la verificación de cardinalidad
filas_previas = len(rendimiento)

# 2. Realizar el left merge declarando validate="many_to_one" e incorporando el indicador
rendimiento_municipal = rendimiento.merge(
    dimension,
    on='codigo_municipio',
    how='left',
    validate='many_to_one',
    indicator='_merge_divipola'
)

# 3. Identificar y mostrar los registros de rendimiento sin correspondencia geográfica en la dimensión
sin_nombre = rendimiento_municipal.query("_merge_divipola == 'left_only'").copy()

print("--- RESULTADOS DE LA UNIÓN (MERGE) ---")
print(f"Filas en 'rendimiento' (antes):          {filas_previas:,}")
print(f"Filas en 'rendimiento_municipal' (después): {len(rendimiento_municipal):,}")
print(f"Códigos sin nombre territorial (sin cruce):  {len(sin_nombre):,}")

if len(sin_nombre) > 0:
    print("\n--- CÓDIGOS MUNICIPIOS SIN NOMBRE TERRITORIAL ---")
    display(sin_nombre[['codigo_municipio', 'Cultivo', 'anio', 'rendimiento_t_ha']])
else:
    print("\n✅ Todos los códigos municipales se vincularon correctamente con su dimensión territorial.")

# 4. Vista previa de la tabla final
print("\n--- VISTA PREVIA DE 'rendimiento_municipal' ---")
display(rendimiento_municipal.head(10))

# 5. Comprobaciones obligatorias (Asserts)
assert len(rendimiento_municipal) == filas_previas, (
    f"Error de cardinalidad: El número de filas cambió de {filas_previas} a {len(rendimiento_municipal)}. "
    "Verifica que 'codigo_municipio' sea único en la dimensión."
)

print("\n✅ COMPROBACIÓN EXITOSA: La cantidad de filas se mantuvo exactamente igual antes y después de la unión.")

--- RESULTADOS DE LA UNIÓN (MERGE) ---
Filas en 'rendimiento' (antes):          292
Filas en 'rendimiento_municipal' (después): 292
Códigos sin nombre territorial (sin cruce):  0

✅ Todos los códigos municipales se vincularon correctamente con su dimensión territorial.

--- VISTA PREVIA DE 'rendimiento_municipal' ---


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t,area_ha,rendimiento_t_ha,nombre_departamento,nombre_municipio,clasificacion_territorial,_merge_divipola
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000,ANTIOQUIA,MEDELLÍN,Municipio,both
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000,ANTIOQUIA,ABEJORRAL,Municipio,both
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000,ANTIOQUIA,ABRIAQUÍ,Municipio,both
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353,ANTIOQUIA,BARBOSA,Municipio,both
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459,ANTIOQUIA,BELMIRA,Municipio,both
5,05088,Papa,En fresco,2024,2,1710.0,95.0,18.000000,ANTIOQUIA,BELLO,Municipio,both
6,05093,Papa,En fresco,2024,1,6.0,2.0,3.000000,ANTIOQUIA,BETULIA,Municipio,both
7,05148,Papa,En fresco,2024,4,14757.0,789.0,18.703422,ANTIOQUIA,EL CARMEN DE VIBORAL,Municipio,both
8,05206,Papa,En fresco,2024,2,1200.0,80.0,15.000000,ANTIOQUIA,CONCEPCIÓN,Municipio,both
9,05237,Papa,En fresco,2024,3,2638.0,103.8,25.414258,ANTIOQUIA,DONMATÍAS,Municipio,both



✅ COMPROBACIÓN EXITOSA: La cantidad de filas se mantuvo exactamente igual antes y después de la unión.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [ ]:
import pandas as pd

# 1. Rutas de exportación en la carpeta OUT
ruta_rendimiento = OUT / "rendimiento_municipal.csv"
ruta_sin_corresp = OUT / "codigos_sin_correspondencia.csv"

# 2. Exportar rendimiento_municipal a CSV sin índice
rendimiento_municipal.to_csv(ruta_rendimiento, index=False)
print(f"✅ Archivo exportado exitosamente: {ruta_rendimiento}")

# 3. Exportar sin_correspondencia a CSV sin índice (conserva cabeceras aunque esté vacío)
sin_correspondencia.to_csv(ruta_sin_corresp, index=False)
print(f"✅ Archivo exportado exitosamente: {ruta_sin_corresp}")

# 4. Comprobación de existencia de los archivos generados
assert ruta_rendimiento.exists(), "Error: No se creó el archivo 'rendimiento_municipal.csv'."
assert ruta_sin_corresp.exists(), "Error: No se creó el archivo 'codigos_sin_correspondencia.csv'."

print("\n✅ COMPROBACIÓN EXITOSA: Ambos archivos CSV se generaron correctamente en la carpeta OUT.")

**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.